# 场景与任务生成

本 Demo 逐项调用场景生成、约束组合、任务编译、任务验证和轨迹规划 API。


## 0. 初始化平台

只定义工作区与输出目录；场景需求、指令和方法参数就地写入 API。


In [ ]:
# ruff: noqa
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("未找到 PhysReal-X 项目根目录。")
    PROJECT_ROOT = PROJECT_ROOT.parent
for import_root in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))
from examples._common import operation_receipt, output_artifact, write_summary
from physrealx import PhysRealX
from physrealx.errors import PhysRealXError

WORKSPACE = PROJECT_ROOT / ".physrealx/examples/03-scenes-tasks"
OUTPUT_DIR = PROJECT_ROOT / "examples/scene_and_task_generation/output"
px = PhysRealX.open(WORKSPACE)
receipts = {}

## 1. 生成场景草案：`scenes.generate`

**用途**：按领域、资产集、机器人和随机种子生成候选布局。

**输出**：`SceneDraft/1`；生成结果只是草案，不能直接进入训练。


In [ ]:
scene_draft_operation = None
scene_draft = None
try:
    scene_draft_operation = px.scenes.generate(
        {
            "domain": "kitchen",
            "seed": 2026,
            "assets": [],
            "robot": "franka.standard@1.0.0",
        },
        method="robocasa365",
        config={},
    )
    scene_draft = output_artifact(scene_draft_operation, "scene_draft")
    receipts["scene_draft"] = operation_receipt(scene_draft_operation)
    print(json.dumps(scene_draft_operation.metrics, ensure_ascii=False, indent=2))
except PhysRealXError as error:
    print(f"[BLOCKED] {error}")

## 2. 约束组合场景：`scenes.compose`

**用途**：对草案执行无穿透、支撑、可达性、工作空间和资产版本约束求解。

**输出**：待人工审核和注册的 `SceneSpec/1`。


In [ ]:
scene_compose_operation = None
if scene_draft is None:
    print("[BLOCKED] 需要先生成场景草案。")
else:
    try:
        scene_compose_operation = px.scenes.compose(
            scene_draft, method="constraint_solver", config={}
        )
        receipts["scene_composition"] = operation_receipt(scene_compose_operation)
        print(json.dumps(scene_compose_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 3. 编译任务谓词：`tasks.compile`

**用途**：把自然语言指令编译成对象绑定、初始条件、目标和确定性成功谓词。

**安全边界**：语言模型只能提出 `TaskSpec` 草案，不直接生成并执行 Python。


In [ ]:
registered_scene = None  # 替换为审核后注册的 SceneSpec/1 VersionedRef
task_compile_operation = None
if registered_scene is None:
    print("[BLOCKED] 请在本单元格绑定已审核 SceneSpec/1。")
else:
    try:
        task_compile_operation = px.tasks.compile(
            "把杯子放进抽屉",
            registered_scene,
            method="bddl",
            config={},
        )
        receipts["task_compilation"] = operation_receipt(task_compile_operation)
        print(json.dumps(task_compile_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 4. 验证任务实例：`tasks.validate`

**用途**：检查对象引用、谓词可满足性、成功条件、操作空间和安全限制。

**输出**：已验证 `TaskInstance/1`，失败样本与成功样本使用同一谓词系统判定。


In [ ]:
registered_task = None  # 替换为审核后注册的 TaskSpec/1 VersionedRef
task_validation_operation = None
if registered_task is None:
    print("[BLOCKED] 请在本单元格绑定已审核 TaskSpec/1。")
else:
    try:
        task_validation_operation = px.tasks.validate(
            registered_task,
            method="task_validator",
            config={},
        )
        receipts["task_validation"] = operation_receipt(task_validation_operation)
        print(json.dumps(task_validation_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 5. 生成轨迹：`trajectories.generate`

**用途**：对已验证任务进行 IK、碰撞检查和轨迹优化，产生可重放的专家轨迹。

**输出**：轨迹、规划指标及失败原因；规划失败不会被改写为成功样本。


In [ ]:
task_instance = None  # 替换为已验证 TaskInstance/1 ArtifactRef
trajectory_operation = None
if task_instance is None:
    print("[BLOCKED] 请在本单元格绑定已验证任务实例。")
else:
    try:
        trajectory_operation = px.trajectories.generate(
            task_instance,
            method="curobo",
            config={"seed": 2026},
        )
        receipts["trajectory_generation"] = operation_receipt(trajectory_operation)
        print(json.dumps(trajectory_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 6. 保存回执

回执区分已发布结果和待审核边界，不自动跨过人工审核。


In [ ]:
report = write_summary(
    OUTPUT_DIR / "scene-task-report.json",
    {"module": "场景与任务生成", "receipts": receipts},
)
print(json.dumps(report, ensure_ascii=False, indent=2, default=str))